## Assignment 1

*100 points (8% of course grade)*</br>
*Assigned: Wed, Sep 16nd*</br>
**Due: Fri, Oct 2nd, 23:59**

This homework should be done in parts as soon as (<= 1 week) relevant topics are covered in lectures. If you wait until the last minute, you might be overwhelmed.

You must turn in the required files electronically, including this Notebook (A1.ipynb) and a few additional files. Please follow the submission instructions for each problem carefully.

In this assignment, you need to solve two problems. In Problem 1, you will write relational algebra queries. In Problem 2, you will draw an E/R diagram.

## Setup environment and test database

You will need this setup if you want to create a database to test whether your answer is correct. Please follow our [setup instructions](https://canvas.sfu.ca/courses/16213/pages/0-main-entrance-general-guidance-on-cmpt-354-environment-setup) on Canvas. We **recommend you finish the setup**, as you can run your queries and debug on your machine, and you will need Postgres in future assignments; **but you are still able to finish Assignment 1 without the setup**.

### Problem 1: Query with Relational Algebra (63%)

Consider a database with following schema. Underlined columns are the keys of the table. 

- drinker (<u>name</u>, address)
- bar (<u>name</u>, address)
- beer (<u>name</u>, brewer)
- frequents (<u>drinker</u>, <u>bar</u>, times_a_week)
- likes (<u>drinker</u>, <u>beer</u>)
- serves (<u>bar</u>, <u>beer</u>, price)


#### **Preliminary**


To write a relational algebra (RA) query in a cell, we have already converted the cells under each question into the [Markdown cell](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html). Then, you will need to type RA queries in the cell using the syntax in [radb](https://users.cs.duke.edu/~junyang/radb/index.html). Basically, it is the same as the Greek letter notation we covered in class after replacing the syntax. Below is an example of an RA query in the radb format. Please refer to [the radb cheatsheet](https://users.cs.duke.edu/~junyang/radb/cheat.html) for the full list of syntax of RA queries. Please use the radb's syntax to ensure we can run your RA queries. Hint: You may draw the query tree on your scratch paper, replace the Greek letter operators with the radb-format operators, and add parentheses.

Example: find the name beers liked by drinkers who frequent the James Joyce Pub bar.

The relational algebra query:

$$\pi_{beer} ( 
        (\sigma_{bar = 'James Joyce Pub'} Frequents)         
         \bowtie_{Frequents.drinker = Likes.drinker} 
         Likes
);$$

In the radb format:
```
\project_{beer} ( 
        (\select_{bar = 'James Joyce Pub'} Frequents)         
         \join_{Frequents.drinker = Likes.drinker}   /* join with Likes to find beers */
         Likes
);
```


You can run RA queries on your local machine after installing radb following the instructions on Canvas. We will also provide an online tool for you to debug your RA queries (also see the instructions on Canvas). It will be much easier to grade if we can run your query and also easier for you to debug with query results. *The use of the tool is optional, and submissions to the online tool will not be graded (**we only grade the Canvas submission**).* *When usng the online tool, since the computation resources are limited, please refrain from submitting too frequently and avoid large joins (like more than three cross products).*



Now your homework question is to write Relational Algebra queries to answer following questions. 

Please fill your answer in each cell (and **ONLY the query**) and **DO NOT add or remove** any cells to make the TAs' life easier in evaluating your queries. Questions (1)-(3) are worth 6 points each; (4)-(6) are worth 7 points each; (7)-(9) are worth 8 points each.


#### 0. (example) Find names of all bars that Eve frequents.

/* input your answer in this cell: */

\project_{bar} (\select_{drinker = 'Eve'} frequents);

#### 1. Find names of beers that  Satisfaction serves

/* input your answer in this cell: */

\project_{beer} (
\select_{bar= 'Satisfaction'} serves
);


#### 2. Find names of bars that Amy frequents more than once a week

/* input your answer in this cell: */

\project_{bar} (
\select_{drinker = 'Amy' and times_a_week > 1} frequents
);


#### 3. Find names of all drinkers who frequent at least two bars

/* input your answer in this cell: */

\project_{drinker1}(
\rename_{d1: drinker1, bar1, times_a_week1} Frequents
\join_{d1.drinker1 = d2.drinker2 and d1.bar1 <> d2.bar2}
\rename_{d2: drinker2, bar2, times_a_week2} Frequents
);


#### 4. Find bars frequented by either Ben or Dan, but not both

/* input your answer in this cell: */

\project_{bar}(
\project_{bar}( 
\select_{drinker = 'Ben' or drinker = 'Dan'} Frequents
)
\diff
\project_{bar}(
\project_{bar}( 
// Bars with Ben and Dan
\select_{drinker = 'Ben'} Frequents
)
\join
\project_{bar}( 
// Bars with Ben and Dan
\select_{drinker = 'Dan'} Frequents
)
)
);

#### 5. Find the names of all drinkers who frequent *every* bar (hint: you may need to use renaming and store your intermediate results using [views](https://users.cs.duke.edu/~junyang/radb/advance.html?highlight=view) to make your querying process more clear.)

e.g. with view, the first example query in preliminary can be written as

```
v1 :- \select_{bar = 'James Joyce Pub'} Frequents;
\project_{beer} ( 
        v1 
         \join_{v1.drinker = Likes.drinker}   /* join with Likes to find beers */
         Likes
);
```
For view names, please only use **lowercase** letters, because ratest may automatically convert letters to lowercase.

/* input your answer in this cell: */

d :- \project_{drinker}(
\rename_{drinker, add} drinker
);
b :- \project_{bar}(
\rename_{bar, add} bar
);
cross :- \project_{drinker, bar}(
d \cross b
);
f :- \project_{drinker, bar}(
\rename_{drinker, bar, times} frequents
);
left :- \project_{drinker} (
cross \diff f
);
d \diff left;

#### 6. Find names and addresses of drinkers who like Corona but do not frequent Satisfaction

/* input your answer in this cell: */

dlc :- \project_{drinker}( // drinkers who like corona
\select_{beer = 'Corona'} likes
);
d :- \project_{drinker}(
\rename_{drinker, add} drinker
);
dfs :- \project_{drinker}( // drinkers who frequent satisfaction
\select_{bar = 'Satisfaction'} frequents
);
dnfs :-\project_{drinker}(// drinkers who don't frequent satisfaction
d \diff dfs
);
n :- \project_{drinker}(// names of drinkers who like corona but don't frequent satisfaction
dlc \intersect dnfs
);

\project_{drinker, address}(
n \join_{name=n.drinker} drinker
);

#### 7. For each beer that Eve likes, find the names of bars that serve it at the lowest price (when a bar serves multiple beers at the same lowest price, they should all be included in the output) (hint: recall the "trickier exercise" in the slides for how to present "the lowest")

/* input your answer in this cell: */

bel :- \project_{beer}( // beer eve likes
\select_{drinker='Eve'} likes
);

places :- \project_{bar, beer, price}(
bel \join serves
);

bts :- \project_{beer, bar} places; // Bars That Serve eve's drinks

right:- \project_{beer1, bar1}(
\rename_{bar1, beer1, price1} places
\join_{ price1>price2 and beer1 = beer2 }
\rename_{bar2, beer2, price2} places
);
bts \diff right;

#### 8. Find names of all drinkers who frequent *only* the bars that serve *some* beers they like (drinkers who frequent no bars are included)


/* input your answer in this cell: */

fb :- \project_{drinker, bar} frequents; // frequented bars, bars the drinker goes to
lb :- \project_{drinker, bar}( // bars that serve min 1 drink the drinker likes, liked bars
likes \join serves 
);
d_out :- \project_{drinker}( // drinkers outside our criteria
fb \diff lb
);
d_in :- \project_{drinker}(
\rename_{drinker, add} drinker
);

d_in \diff d_out;

#### 9. For each beer, find the drinkers who like this beer but frequent *none* of the bars serving this beer. Format your output as a list of (beer, drinker) pairs.

/* input your answer in this cell: */

set :- \project_{beer, drinker} likes;
sb :- \project_{bar, beer}(
set \join serves
);
fb :- \project_{bar, beer}(
set \join frequents
);
rb :- fb \intersect sb; // bars they like a drink of and frequent
// so now anyone who frequents these bars in rb are d_out
d_out :-\project_{beer, drinker}(
frequents \join rb
);
set \diff d_out;

## Problem 2: ER design (37%)

Design a database that captures the following information:
    
    
- Each person is either a student or teacher, but not both.


- Each person has a unique ID, a name, and phone (denoted by the model). 


- The university offers different courses of study. Each course has a unique name and belongs to a department. In any given school year, a given subject can be taught by only one teacher. A course can be taught over multiple years and a student may study the same course multiple times.


- For each student, you need to additionally record the year when he or she entered the university (the class year), as well as his or her favorite subjects.


- A student or a teacher can belong to one or multiple departments. You should be able to track each department's head and its current students.


- Each department has multiple clubs. You should be able to track each club's current students.


Design an E/R diagram for this database. Replace the current `ER-diagram.png` with your figure (you may also use other picture formats like .jpg, but remember to change the filename in the cell below). If you prefer a web-based tool, you may use drawio (https://app.diagrams.net/) for building your E/R diagram. Very briefly explain the intuitive meaning of any entity and relationship sets as needed. Do not forget to indicate keys and multiplicity of relationships, as well as ISA relationships and weak entity sets (if any), using appropriate notation.


If you think some aspects of the above are unclear, feel free to make additional, reasonable assumptions, but state them clearly in your answer. Also, keep in mind that there is no single “correct” design.

<img src="ER-diagram.png" alt="Drawing" style="width: 800px;"/>

`Write a brief explanation for your design in this cell`

I'll make the assumption that subjects == courses.
You can track the # of students in a department using RA filters, same with clubs, 
and same for teachers in departments. Underlined attributes are keys, I've decided to make 'unique'
attributes keys (e.g., the problem states that each course has a unique name, so 'name' is a key for course).

EXPLAIN MORE AFTER SEEING TA/PROF


### Submission instruction

1. For problem 1, answer the questions (1)-(9) in the Markdown cells.

2. For problem 2, replace `ER-diagram.png` with your ER diagram in a png/jpg file; write some explanation in the Markdown cell.

3. Compress your A1.ipynb (this file) and your ER diagram into A1.zip and submit on Canvas.